**# 05 — Gold Layer
Silver → Gold: builds dim_date, dim_location and fact_weather_daily (star schema)**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime
from delta.tables import DeltaTable

try:
    _ = LOAD_TYPE
except NameError:
    dbutils.widgets.text("load_type", "full", "Load Type (full/incremental)")
    LOAD_TYPE = dbutils.widgets.get("load_type")

try:
    _ = BATCH_ID
except NameError:
    dbutils.widgets.text("batch_id", "batch_fullload_2000_2026", "Batch ID")
    BATCH_ID = dbutils.widgets.get("batch_id")

SILVER_TABLE        = "pakistan_weather.silver_weather_cleaned"
DIM_DATE            = "pakistan_weather.dim_date"
DIM_LOCATION        = "pakistan_weather.dim_location"
FACT_HOURLY         = "pakistan_weather.fact_weather_hourly"
AGG_MONTHLY         = "pakistan_weather.agg_monthly_climate"
AGG_YEARLY_EXTREMES = "pakistan_weather.agg_yearly_extremes"

# ── RUN STATUS (carried from cell to cell) ──
gold_status     = "Success"
gold_error      = None
rows_aggregated = 0
rows_inserted   = 0
rows_updated    = 0
start_time      = datetime.now()

print(f"Load Type : {LOAD_TYPE}")
print(f"Batch ID  : {BATCH_ID}")

In [0]:
if gold_status == "Success":
    try:
        CITIES = [
            ("Lahore",    "Punjab",                      31.5204, 74.3587),
            ("Karachi",   "Sindh",                       24.8607, 67.0011),
            ("Islamabad", "Islamabad Capital Territory", 33.6844, 73.0479),
            ("Peshawar",  "Khyber Pakhtunkhwa",          34.0151, 71.5249),
            ("Quetta",    "Balochistan",                 30.1798, 66.9750),
        ]
        df_cities = spark.createDataFrame(CITIES, ["city", "province", "city_latitude", "city_longitude"])
        key_map   = spark.createDataFrame([(i + 1, c[0]) for i, c in enumerate(CITIES)], ["location_key", "city"])

        df_grid = spark.table(SILVER_TABLE).select("latitude", "longitude").distinct() \
            .withColumnRenamed("latitude", "grid_latitude") \
            .withColumnRenamed("longitude", "grid_longitude")

        w = Window.partitionBy("grid_latitude", "grid_longitude").orderBy("dist")

        df_dim_location = (df_grid.crossJoin(df_cities)
            .withColumn("dist", (F.col("grid_latitude") - F.col("city_latitude")) ** 2 +
                                (F.col("grid_longitude") - F.col("city_longitude")) ** 2)
            .withColumn("rn", F.row_number().over(w))
            .filter("rn = 1")
            .join(key_map, "city")
            .select("location_key", "city", "province", "grid_latitude", "grid_longitude",
                    "city_latitude", "city_longitude"))

        df_dim_location.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(DIM_LOCATION)

        n_loc = spark.table(DIM_LOCATION).count()
        print(f"dim_location written: {n_loc} rows")
        if n_loc != len(CITIES):
            raise Exception(f"Expected {len(CITIES)} locations but found {n_loc}. Check Silver latitude/longitude values.")
    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"GOLD FAILED: {e}")

In [0]:
if gold_status == "Success":
    try:
        # 1. Inspect Silver to find the exact max observation year currently loaded
        max_silver_year = spark.table(SILVER_TABLE) \
            .select(F.max(F.year("observation_timestamp")).alias("max_yr")) \
            .collect()[0]["max_yr"]

        # 2. Set max_year directly to max_silver_year (fallback to current year if table is empty)
        max_year = max_silver_year if max_silver_year is not None else datetime.now().year

        # 3. Dynamically generate dates from 2000-01-01 through {max_year}-12-31
        df_dates = spark.sql(f"""
            SELECT explode(sequence(to_date('2000-01-01'), to_date('{max_year}-12-31'), interval 1 day)) AS full_date
        """)

        # 4. Derive calendar hierarchy and temporal features
        df_dim_date = (df_dates
            .withColumn("date_key",    F.date_format("full_date", "yyyyMMdd").cast("int"))
            .withColumn("year",        F.year("full_date"))
            .withColumn("quarter",     F.quarter("full_date"))
            .withColumn("month",       F.month("full_date"))
            .withColumn("month_name",  F.date_format("full_date", "MMMM"))
            .withColumn("day",         F.dayofmonth("full_date"))
            .withColumn("day_of_week", F.dayofweek("full_date"))
            .withColumn("day_name",    F.date_format("full_date", "EEEE"))
            .withColumn("is_weekend",  F.dayofweek("full_date").isin(1, 7))
            .withColumn("season", F.when(F.col("month").isin(3, 4, 5),   "Spring")
                                   .when(F.col("month").isin(6, 7, 8),   "Summer")
                                   .when(F.col("month").isin(9, 10, 11), "Autumn")
                                   .otherwise("Winter"))
            .withColumn("is_monsoon",  F.col("month").isin(7, 8, 9))
            .select("date_key", "full_date", "year", "quarter", "month", "month_name",
                    "day", "day_of_week", "day_name", "is_weekend", "season", "is_monsoon"))

        # 5. Overwrite dim_date table with the updated calendar
        df_dim_date.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true").saveAsTable(DIM_DATE)

        print(f"dim_date written up to {max_year}-12-31: {spark.table(DIM_DATE).count():,} rows")
    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"GOLD FAILED: {e}")

In [0]:
if gold_status == "Success":
    try:
        df_silver = spark.table(SILVER_TABLE)

        
        df_silver = df_silver.filter(F.col("batch_id") == BATCH_ID)

        dim_loc  = spark.table(DIM_LOCATION)
        dim_date = spark.table(DIM_DATE).select("date_key", "full_date")

        # Join Silver to Dimensions & compute derived features (time_of_day, heat_index)
        df_fact_hourly = (
            df_silver
            .join(F.broadcast(dim_loc),
                  (df_silver.latitude == dim_loc.grid_latitude) & 
                  (df_silver.longitude == dim_loc.grid_longitude))
            .withColumn("full_date", F.to_date("observation_timestamp"))
            .join(F.broadcast(dim_date), "full_date")
            # Derive time_of_day
            .withColumn("time_of_day", 
                F.when((F.col("hour") >= 5) & (F.col("hour") < 12), "Morning")
                 .when((F.col("hour") >= 12) & (F.col("hour") < 17), "Afternoon")
                 .when((F.col("hour") >= 17) & (F.col("hour") < 21), "Evening")
                 .otherwise("Night")
            )
            # Derive heat_index (approximation formula for temps >= 27°C)
            .withColumn("heat_index",
                F.when(F.col("temperature_2m") >= 27,
                    F.round(
                        -8.78469475556 + 
                        1.61139411 * F.col("temperature_2m") + 
                        2.33854883889 * F.col("relative_humidity_2m") - 
                        0.14611605 * F.col("temperature_2m") * F.col("relative_humidity_2m") - 
                        0.012308094 * (F.col("temperature_2m")**2) - 
                        0.0164248277778 * (F.col("relative_humidity_2m")**2) + 
                        0.002211732 * (F.col("temperature_2m")**2) * F.col("relative_humidity_2m") + 
                        0.00072546 * F.col("temperature_2m") * (F.col("relative_humidity_2m")**2) - 
                        0.000003582 * (F.col("temperature_2m")**2) * (F.col("relative_humidity_2m")**2), 
                    2)
                ).otherwise(F.col("temperature_2m"))
            )
            .select(
                "location_key",
                "date_key",
                "observation_timestamp",
                "hour",
                "time_of_day",
                "temperature_2m",
                "apparent_temperature",
                "heat_index",
                "relative_humidity_2m",
                "precipitation",
                "rain",
                "wind_speed_10m",
                "wind_gusts_10m",
                "pressure_msl",
                "cloud_cover",
                "shortwave_radiation",
                "et0_fao_evapotranspiration"
            )
            .withColumn("gold_load_timestamp", F.current_timestamp())
            .withColumn("batch_id", F.lit(BATCH_ID))
        )

        rows_aggregated = df_fact_hourly.count()
        df_fact_hourly.createOrReplaceTempView("hourly_updates")
        print(f"Hourly updates view created successfully for batch '{BATCH_ID}'. Rows: {rows_aggregated:,}")

    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"GOLD HOURLY PREP FAILED: {e}")

In [0]:
if gold_status == "Success":
    try:
        if not spark.catalog.tableExists(FACT_HOURLY):
            spark.sql(f"""
                CREATE TABLE {FACT_HOURLY}
                USING DELTA
                CLUSTER BY (location_key, date_key)
                AS SELECT * FROM hourly_updates WHERE 1=0
            """)
            print(f"{FACT_HOURLY} created (clustered by location_key, date_key)")
        
        rows_aggregated = df_fact_hourly.count()
        spark.sql(f"""
            MERGE INTO {FACT_HOURLY} AS target
            USING hourly_updates AS source
            ON  target.location_key          = source.location_key
            AND target.observation_timestamp = source.observation_timestamp
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        metrics = DeltaTable.forName(spark, FACT_HOURLY).history(1).collect()[0]["operationMetrics"]
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

        print(f"fact_weather_hourly MERGE done. Inserted: {rows_inserted:,}  Updated: {rows_updated:,}")
    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"GOLD HOURLY MERGE FAILED: {e}")

In [0]:
if gold_status == "Success":
    try:
        # Aggregate monthly climate metrics per city with date_key pointing to month start (YYYYMM01)
        df_monthly = (
            spark.table(FACT_HOURLY)
            .join(spark.table(DIM_DATE), "date_key")
            .groupBy("location_key", "year", "month", "month_name")
            .agg(
                F.round(F.avg("temperature_2m"), 2).alias("avg_temp_c"),
                F.round(F.min("temperature_2m"), 2).alias("min_temp_c"),
                F.round(F.max("temperature_2m"), 2).alias("max_temp_c"),
                F.round(F.sum("precipitation"), 2).alias("total_precip_mm"),
                F.round(F.avg("wind_speed_10m"), 2).alias("avg_wind_speed_kmh"),
                F.count("*").alias("hours_observed")
            )
            .withColumn("date_key", (F.col("year") * 10000 + F.col("month") * 100 + 1).cast("int"))
            .withColumn("gold_load_timestamp", F.current_timestamp())
            .withColumn("batch_id", F.lit(BATCH_ID))
        )

        df_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(AGG_MONTHLY)
        print(f"agg_monthly_climate written: {spark.table(AGG_MONTHLY).count():,} rows")

    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"AGG MONTHLY CLIMATE FAILED: {e}")

In [0]:
if gold_status == "Success":
    try:
        df_daily_extremes = (
            spark.table(FACT_HOURLY)
            .join(spark.table(DIM_DATE), "date_key")
            .groupBy("location_key", "year", "full_date")
            .agg(
                F.max("temperature_2m").alias("daily_max_temp"),
                F.min("temperature_2m").alias("daily_min_temp"),
                F.sum("precipitation").alias("daily_precip"),
                F.max("wind_speed_10m").alias("daily_max_wind")
            )
        )

        df_yearly = (
            df_daily_extremes
            .groupBy("location_key", "year")
            .agg(
                F.max("daily_max_temp").alias("hottest_temp_c"),
                F.min("daily_min_temp").alias("coldest_temp_c"),
                F.round(F.max("daily_precip"), 2).alias("wettest_day_precip_mm"),
                F.max("daily_max_wind").alias("highest_wind_speed_kmh")
            )
            .withColumn("date_key", (F.col("year") * 10000 + 101).cast("int"))
            .withColumn("gold_load_timestamp", F.current_timestamp())
            .withColumn("batch_id", F.lit(BATCH_ID))
        )

        df_yearly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(AGG_YEARLY_EXTREMES)
        print(f"agg_yearly_extremes written: {spark.table(AGG_YEARLY_EXTREMES).count():,} rows")

    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"AGG YEARLY EXTREMES FAILED: {e}")

In [0]:
if gold_status == "Success" and LOAD_TYPE == "full":
    try:
        spark.sql(f"OPTIMIZE {FACT_HOURLY}").show(truncate=False)
        spark.sql(f"DESCRIBE DETAIL {FACT_HOURLY}") \
            .select("numFiles", "sizeInBytes", "clusteringColumns").show(truncate=False)
    except Exception as e:
        gold_status = "Failure"
        gold_error  = str(e)
        print(f"GOLD OPTIMIZE FAILED: {e}")
#run this only on a full load, so daily incremental runs don’t pay for compaction. That’s also a FinOps point for report.

In [0]:
# ── DATA QUALITY & INTEGRITY VALIDATION CHECKS ──
print("Running automated data quality checks on Gold layer...\n")

# 1. Print row counts for all Gold tables
gold_tables = [DIM_LOCATION, DIM_DATE, FACT_HOURLY, AGG_MONTHLY, AGG_YEARLY_EXTREMES]
for t in gold_tables:
    print(f"{t:<45} {spark.table(t).count():>12,} rows")

print("\n" + "="*60 + "\n")

# 2. Validate exact location dimension count (Expects 5 Pakistan cities)
loc_count = spark.table(DIM_LOCATION).count()
assert loc_count == 5, f"Validation Failed: Expected exactly 5 locations in {DIM_LOCATION}, found {loc_count}."

# 3. Primary Key Uniqueness Check on Fact Table
dup_keys = (
    spark.table(FACT_HOURLY)
    .groupBy("location_key", "observation_timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
assert dup_keys == 0, f"Validation Failed: Found {dup_keys} duplicate key combinations in {FACT_HOURLY}."

# 4. Referential Integrity Checks (Orphan Keys)
orphan_cities = (
    spark.table(FACT_HOURLY)
    .select("location_key")
    .distinct()
    .join(spark.table(DIM_LOCATION).select("location_key"), "location_key", "left_anti")
    .count()
)
assert orphan_cities == 0, f"Validation Failed: Found {orphan_cities} orphan location_keys in {FACT_HOURLY}."

orphan_dates = (
    spark.table(FACT_HOURLY)
    .select("date_key")
    .distinct()
    .join(spark.table(DIM_DATE).select("date_key"), "date_key", "left_anti")
    .count()
)
assert orphan_dates == 0, f"Validation Failed: Found {orphan_dates} orphan date_keys in {FACT_HOURLY}."

print("All automated data quality & integrity checks passed successfully!")
print("\n" + "="*60 + "\n")

In [0]:
end_time = datetime.now()

log_schema = StructType([
    StructField("layer",           StringType(),    True),
    StructField("load_type",       StringType(),    True),
    StructField("file_processed",  StringType(),    True),
    StructField("city",            StringType(),    True),
    StructField("start_time",      TimestampType(), True),
    StructField("end_time",        TimestampType(), True),
    StructField("status",          StringType(),    True),
    StructField("rows_inserted",   LongType(),      True),
    StructField("rows_updated",    LongType(),      True),
    StructField("rows_aggregated", LongType(),      True),
    StructField("error_message",   StringType(),    True),
    StructField("batch_id",        StringType(),    True),
])

log_data = [{
    "layer":           "Silver-to-Gold",
    "load_type":       LOAD_TYPE,
    "file_processed":  SILVER_TABLE,
    "city":            "All-5-Cities",
    "start_time":      start_time,
    "end_time":        end_time,
    "status":          gold_status,
    "rows_inserted":   rows_inserted,
    "rows_updated":    rows_updated,
    "rows_aggregated": rows_aggregated,
    "error_message":   gold_error,
    "batch_id":        BATCH_ID,
}]

log_df = spark.createDataFrame(log_data, schema=log_schema)
log_df.write.format("delta").mode("append").saveAsTable("pakistan_weather.pipeline_execution_logs")

if gold_status == "Failure":
    raise Exception(f"Gold failed: {gold_error}")

print(f"Rows aggregated : {rows_aggregated:,}")
print(f"Rows inserted   : {rows_inserted:,}")
print(f"Rows updated    : {rows_updated:,}")
print(f"\n{'='*45}")
print(f"GOLD COMPLETE!  Status: {gold_status}  Duration: {(end_time-start_time).seconds}s")
print(f"{'='*45}")

In [0]:
# 1. Print row counts for all Gold tables
for t in ["dim_date", "dim_location", "fact_weather_hourly", "agg_monthly_climate", "agg_yearly_extremes"]:
    print(f"{t:<25} : {spark.table(f'pakistan_weather.{t}').count():,} rows")

print("\n" + "="*60 + "\n")

# 2. Verify hourly fact table coverage and average temperatures per city
spark.sql("""
    SELECT 
        l.city, 
        MIN(d.full_date) AS first_day, 
        MAX(d.full_date) AS last_day, 
        COUNT(*) AS total_hours_observed,
        ROUND(AVG(f.temperature_2m), 1) AS avg_temp_c
    FROM pakistan_weather.fact_weather_hourly f
    JOIN pakistan_weather.dim_location l ON f.location_key = l.location_key
    JOIN pakistan_weather.dim_date d     ON f.date_key = d.date_key
    GROUP BY l.city 
    ORDER BY l.city
""").show()

# 3. Verify aggregate tables match expected full load counts
print("Monthly Climate Summary Rows per City:")
spark.sql("""
    SELECT l.city, COUNT(*) AS monthly_records
    FROM pakistan_weather.agg_monthly_climate a
    JOIN pakistan_weather.dim_location l ON a.location_key = l.location_key
    GROUP BY l.city ORDER BY l.city
""").show()

print("Yearly Extremes Rows per City:")
spark.sql("""
    SELECT l.city, COUNT(*) AS yearly_records
    FROM pakistan_weather.agg_yearly_extremes a
    JOIN pakistan_weather.dim_location l ON a.location_key = l.location_key
    GROUP BY l.city ORDER BY l.city
""").show()